In [ ]:
import sys
import torch
import pickle
import numpy as np
import os

# Force immediate output
sys.stdout.flush()

print("🚀 Starting BiLSTM evaluation...", flush=True)

# ===========================
# Model Definition
# ===========================
class BiLSTMRegressor(torch.nn.Module):
    def __init__(self, feature_dim, hidden_size, num_layers, bidirectional, company_count, company_emb_dim, dropout=0.0):
        super().__init__()
        self.company_emb = torch.nn.Embedding(company_count, company_emb_dim)
        rnn_input_dim = feature_dim + company_emb_dim
        self.lstm = torch.nn.LSTM(
            input_size=rnn_input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        out_dim = hidden_size * (2 if bidirectional else 1)
        self.head = torch.nn.Sequential(
            torch.nn.Linear(out_dim, out_dim // 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(out_dim // 2, 1)
        )

    def forward(self, x, c):
        b, s, f = x.shape
        c_emb = self.company_emb(c).unsqueeze(1).expand(-1, s, -1)
        rnn_in = torch.cat([x, c_emb], dim=-1)
        out, _ = self.lstm(rnn_in)
        last = out[:, -1, :]
        return self.head(last).squeeze(-1)

# ===========================
# Step 1: Load Checkpoint
# ===========================
print("Step 1: Loading checkpoint...", flush=True)
SAVE_PATH = "./bilstm_company_model.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"  Device: {DEVICE}", flush=True)

if not os.path.exists(SAVE_PATH):
    print(f"❌ Checkpoint file not found at {SAVE_PATH}", flush=True)
    print(f"Files in current directory: {os.listdir('.')}", flush=True)
    exit(1)

try:
    checkpoint = torch.load(SAVE_PATH, map_location=DEVICE)
    print(f"✅ Checkpoint loaded. Keys: {list(checkpoint.keys())}", flush=True)
except Exception as e:
    print(f"❌ Error loading checkpoint: {e}", flush=True)
    exit(1)

mean = checkpoint.get("mean")
std = checkpoint.get("std")
company_list = checkpoint.get("company_list") or checkpoint.get("enc_classes")
feature_dim = checkpoint.get("feature_dim")

print(f"  mean shape: {mean.shape if mean is not None else None}", flush=True)
print(f"  std shape: {std.shape if std is not None else None}", flush=True)
print(f"  companies: {len(company_list) if company_list else 0}", flush=True)
print(f"  feature_dim: {feature_dim}", flush=True)

if any(x is None for x in [mean, std, company_list, feature_dim]):
    print("❌ Missing required checkpoint keys", flush=True)
    exit(1)

print("✅ Checkpoint OK", flush=True)

# ===========================
# Step 2: Check Data Folder
# ===========================
print("\nStep 2: Checking data folder...", flush=True)
company_folder = "/home/sunkari/Stock_price_predictor/Company_Specific/windows"

if not os.path.exists(company_folder):
    print(f"❌ Folder not found: {company_folder}", flush=True)
    exit(1)

files = [f for f in os.listdir(company_folder) if f.endswith(".pkl")]
print(f"✅ Found {len(files)} pkl files", flush=True)
if len(files) > 0:
    print(f"  First 5 files: {files[:5]}", flush=True)

# ===========================
# Step 3: Load Test Data
# ===========================
print("\nStep 3: Loading test data...", flush=True)

from sklearn.preprocessing import LabelEncoder

enc = LabelEncoder()
enc.fit(company_list)
print(f"  Encoder fitted with {len(enc.classes_)} companies", flush=True)

all_windows = []
processed = 0

for file in sorted(os.listdir(company_folder))[:5]:  # LIMIT to first 5 files for debugging
    if not file.endswith(".pkl") or "scaler" in file.lower():
        continue
    
    print(f"  Processing: {file}...", flush=True)
    
    try:
        company_name = file.split("_stock")[0]
        with open(os.path.join(company_folder, file), "rb") as f:
            data = pickle.load(f)
        
        if not isinstance(data, tuple) or len(data) != 2:
            print(f"    ⚠️ Skipped (bad format)", flush=True)
            continue
        
        X_array, y_array = data
        print(f"    X shape: {X_array.shape}, y shape: {y_array.shape}", flush=True)
        
        if len(y_array.shape) > 1:
            y_array = y_array[:, 0]
        
        if company_name not in enc.classes_:
            print(f"    ⚠️ Company not in encoder", flush=True)
            continue
        
        company_idx = int(enc.transform([company_name])[0])
        
        for i in range(min(5, len(X_array))):  # LIMIT to first 5 per file
            all_windows.append((X_array[i], float(y_array[i]), company_idx))
        
        processed += 1
        print(f"    ✅ Added {min(5, len(X_array))} windows", flush=True)
    
    except Exception as e:
        print(f"    ❌ Error: {e}", flush=True)
        continue

print(f"✅ Loaded {len(all_windows)} total windows from {processed} files", flush=True)

if len(all_windows) == 0:
    print("❌ No windows loaded!", flush=True)
    exit(1)

# ===========================
# Step 4: Create Test Loader
# ===========================
print("\nStep 4: Creating test loader...", flush=True)

split_idx = int(0.8 * len(all_windows))
test_windows = all_windows[split_idx:]
print(f"  Test windows: {len(test_windows)}", flush=True)

def normalize_window(X):
    return np.nan_to_num((X - mean) / std, nan=0.0).astype(np.float32)

from torch.utils.data import Dataset, DataLoader

class WindowDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows
    def __len__(self):
        return len(self.windows)
    def __getitem__(self, idx):
        X, y, c = self.windows[idx]
        return (torch.tensor(normalize_window(X), dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32),
                torch.tensor(c, dtype=torch.long))

test_loader = DataLoader(WindowDataset(test_windows), batch_size=16, shuffle=False)
print(f"✅ Test loader ready: {len(test_loader)} batches", flush=True)

# ===========================
# Step 5: Build and Load Model
# ===========================
print("\nStep 5: Building model...", flush=True)

model = BiLSTMRegressor(
    feature_dim=feature_dim,
    hidden_size=128,
    num_layers=3,
    bidirectional=True,
    company_count=len(company_list),
    company_emb_dim=32,
    dropout=0.3
).to(DEVICE)

print(f"✅ Model created", flush=True)

try:
    model.load_state_dict(checkpoint["model_state"])
    print(f"✅ Weights loaded", flush=True)
except Exception as e:
    print(f"❌ Error loading weights: {e}", flush=True)
    exit(1)

model.eval()
print(f"✅ Model in eval mode", flush=True)

# ===========================
# Step 6: Evaluate
# ===========================
print("\nStep 6: Evaluating...", flush=True)

true_vals, preds = [], []

with torch.no_grad():
    for i, (X, y, c) in enumerate(test_loader):
        print(f"  Batch {i}...", flush=True)
        X, c = X.to(DEVICE), c.to(DEVICE)
        pred = model(X, c).cpu().numpy()
        preds.extend(pred)
        true_vals.extend(y.numpy())

print(f"✅ Got {len(preds)} predictions", flush=True)

# ===========================
# Step 7: Metrics
# ===========================
print("\nStep 7: Computing metrics...", flush=True)

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(true_vals, preds)
mae = mean_absolute_error(true_vals, preds)
rmse = np.sqrt(mse)
r2 = r2_score(true_vals, preds)

print(f"\n📊 Test Results:")
print(f"  MSE  = {mse:.6f}", flush=True)
print(f"  RMSE = {rmse:.6f}", flush=True)
print(f"  MAE  = {mae:.6f}", flush=True)
print(f"  R²   = {r2:.6f}", flush=True)
print(f"\n✅ Done!", flush=True)

In [ ]:
# bilstm_eval_fixed.py
import sys
import os
import time
import torch
import pickle
import numpy as np

# Force immediate output in many environments
sys.stdout.flush()

def info(msg):
    print(msg, flush=True)

info("🚀 Starting BiLSTM evaluation...")

# ===========================
# Model Definition
# ===========================
class BiLSTMRegressor(torch.nn.Module):
    def __init__(self, feature_dim, hidden_size, num_layers, bidirectional, company_count, company_emb_dim, dropout=0.0):
        super().__init__()
        self.company_emb = torch.nn.Embedding(company_count, company_emb_dim)
        rnn_input_dim = feature_dim + company_emb_dim
        self.lstm = torch.nn.LSTM(
            input_size=rnn_input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        out_dim = hidden_size * (2 if bidirectional else 1)
        self.head = torch.nn.Sequential(
            torch.nn.Linear(out_dim, out_dim // 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(out_dim // 2, 1)
        )

    def forward(self, x, c):
        b, s, f = x.shape
        c_emb = self.company_emb(c).unsqueeze(1).expand(-1, s, -1)
        rnn_in = torch.cat([x, c_emb], dim=-1)
        out, _ = self.lstm(rnn_in)
        last = out[:, -1, :]
        return self.head(last).squeeze(-1)

# ===========================
# Config / Paths
# ===========================
SAVE_PATH = "./bilstm_company_model.pt"
company_folder = "/home/sunkari/Stock_price_predictor/Company_Specific/windows"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
info(f"Device: {DEVICE}")

# ===========================
# Step 1: Load Checkpoint
# ===========================
info("\nStep 1: Loading checkpoint...")

if not os.path.exists(SAVE_PATH):
    info(f"❌ Checkpoint file not found at {SAVE_PATH}")
    info(f"Files in current directory: {os.listdir('.')}")
    raise SystemExit(1)

try:
    checkpoint = torch.load(SAVE_PATH, map_location=DEVICE)
    info(f"✅ Checkpoint loaded. Keys: {list(checkpoint.keys())}")
except Exception as e:
    info(f"❌ Error loading checkpoint: {e}")
    raise SystemExit(1)

# Extract required items from checkpoint (handle multiple key names)
mean = checkpoint.get("mean")
std = checkpoint.get("std")
company_list = checkpoint.get("company_list") or checkpoint.get("enc_classes") or checkpoint.get("enc_classes_list")
feature_dim = checkpoint.get("feature_dim") or checkpoint.get("feature_dims")  # fallback keys if any

info(f"  mean shape: {getattr(mean, 'shape', None)}")
info(f"  std shape: {getattr(std, 'shape', None)}")
info(f"  companies: {len(company_list) if company_list is not None else 0}")
info(f"  feature_dim: {feature_dim}")

if any(x is None for x in [mean, std, company_list, feature_dim]):
    info("❌ Missing required checkpoint keys (mean, std, company_list, feature_dim). Check your checkpoint.")
    raise SystemExit(1)

# ensure company_list is a Python list of strings
if isinstance(company_list, np.ndarray):
    company_list = company_list.tolist()
company_list = [str(x) for x in company_list]

# ===========================
# Step 2: Check data folder
# ===========================
info("\nStep 2: Checking data folder...")
if not os.path.exists(company_folder):
    info(f"❌ Folder not found: {company_folder}")
    raise SystemExit(1)

files_all = sorted([f for f in os.listdir(company_folder) if f.endswith(".pkl")])
info(f"✅ Found {len(files_all)} pkl files in {company_folder}")
if len(files_all) > 0:
    info(f"  First 8 files: {files_all[:8]}")

# ===========================
# Step 3: Load test data (robust)
# ===========================
info("\nStep 3: Loading windows from .pkl files (robust mode)...")

from sklearn.preprocessing import LabelEncoder
enc = LabelEncoder().fit(company_list)
info(f"  Encoder fitted with {len(enc.classes_)} companies")

all_windows = []
files_processed = 0
skipped_files = 0
total_windows_added = 0

for file in files_all:
    if "scaler" in file.lower():
        # skip scaler files if present
        continue
    path = os.path.join(company_folder, file)
    info(f"  Processing {file} ...")
    try:
        with open(path, "rb") as f:
            data = pickle.load(f)
    except Exception as e:
        info(f"    ❌ Could not load pickle: {e} (skipping file)")
        skipped_files += 1
        continue

    # Accept both tuple-of-2: (X_array, y_array) 
    # and list/array of windows with (X,y) or (X,y,t)
    if isinstance(data, tuple) and len(data) == 2:
        X_array, y_array = data
        # ensure arrays
        X_array = np.asarray(X_array)
        y_array = np.asarray(y_array)

        # normalize y_array shape
        if y_array.ndim > 1 and y_array.shape[1] > 1:
            y_array = y_array[:, 0]

        # company name from filename
        company_name = file.split("_stock")[0]
        # try to match encoder classes robustly
        if company_name not in enc.classes_:
            # attempt basic matches (strip suffixes)
            candidate = company_name.split(".")[0]
            if candidate in enc.classes_:
                company_name = candidate
            else:
                # try uppercase/lowercase match
                candidate2 = company_name.upper()
                if candidate2 in enc.classes_:
                    company_name = candidate2

        if company_name not in enc.classes_:
            info(f"    ⚠️ Company '{company_name}' not in checkpoint company list — skipping file")
            skipped_files += 1
            continue

        company_idx = int(enc.transform([company_name])[0])

        n_add = min(len(X_array), len(y_array))
        for i in range(n_add):
            try:
                all_windows.append((np.asarray(X_array[i]), float(y_array[i]), company_idx))
            except Exception as e:
                # skip malformed rows
                continue

        files_processed += 1
        total_windows_added += n_add
        info(f"    ✅ Added {n_add} windows from {file}")

    else:
        # other formats: try to iterate if it's a list of windows
        try:
            added = 0
            for entry in data:
                # possible entry formats:
                # (X, y), ((X,y), t), (X, y, t), or dict with keys
                X = y = t = None
                if isinstance(entry, dict):
                    X = entry.get("X") or entry.get("x") or entry.get("window")
                    y = entry.get("y") or entry.get("target")
                    t = entry.get("t") or entry.get("ticker") or entry.get("company")
                elif isinstance(entry, tuple) or isinstance(entry, list):
                    if len(entry) == 3:
                        X, y, t = entry
                    elif len(entry) == 2:
                        # could be ((X,y), t) or (X, y)
                        a, b = entry
                        if isinstance(a, (tuple, list)) and len(a) == 2:
                            X, y = a
                            t = b
                        else:
                            X, y = a, b
                            t = file.split("_stock")[0]  # fallback
                    else:
                        # unexpected length: try best-effort
                        X = entry[0]
                        y = entry[1] if len(entry) > 1 else None
                        t = entry[2] if len(entry) > 2 else file.split("_stock")[0]
                else:
                    # cannot handle this entry type
                    continue

                if X is None or y is None:
                    continue

                # normalize types
                X = np.asarray(X)
                y = float(np.asarray(y).flat[0])
                # determine company index
                if t is None:
                    t_name = file.split("_stock")[0]
                else:
                    t_name = str(t).split("_stock")[0] if isinstance(t, str) else str(t)

                if t_name not in enc.classes_:
                    # skip if ticker not known
                    continue

                c_idx = int(enc.transform([t_name])[0])
                all_windows.append((X, y, c_idx))
                added += 1

            if added > 0:
                files_processed += 1
                total_windows_added += added
                info(f"    ✅ Added {added} windows (iterable format) from {file}")
            else:
                info(f"    ⚠️ No valid windows found inside {file} (iterable)")
                skipped_files += 1
        except Exception as e:
            info(f"    ❌ Unexpected format inside file: {e} (skipping)")
            skipped_files += 1
            continue

info(f"\n✅ Loaded windows from {files_processed} files, skipped {skipped_files} files. Total windows collected: {len(all_windows)} (counted {total_windows_added})")

if len(all_windows) == 0:
    info("❌ No windows loaded — nothing to evaluate. Exiting.")
    raise SystemExit(1)

# ===========================
# Step 4: Create test loader
# ===========================
info("\nStep 4: Creating DataLoader for evaluation...")

# We'll use all loaded windows as the test set here.
test_windows = all_windows

mean = np.asarray(mean)
std = np.asarray(std)

def normalize_window(X):
    X = np.asarray(X)
    return np.nan_to_num((X - mean) / std, nan=0.0).astype(np.float32)

from torch.utils.data import Dataset, DataLoader

class WindowDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows
    def __len__(self):
        return len(self.windows)
    def __getitem__(self, idx):
        X, y, c = self.windows[idx]
        return (torch.tensor(normalize_window(X), dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32),
                torch.tensor(c, dtype=torch.long))

# Use single-process loader to avoid multiprocessing deadlocks
test_loader = DataLoader(WindowDataset(test_windows), batch_size=16, shuffle=False, num_workers=0)
info(f"✅ Test loader ready: {len(test_loader)} batches (batch_size=16, num_workers=0)")

# Sanity: pull one batch
try:
    xb, yb, cb = next(iter(test_loader))
    info(f"Sanity check batch shapes: X={xb.shape}, y={yb.shape}, c={cb.shape}")
except Exception as e:
    info(f"❌ Error creating first batch from DataLoader: {e}")
    raise SystemExit(1)

# ===========================
# Step 5: Build model and load weights
# ===========================
info("\nStep 5: Building model...")

model = BiLSTMRegressor(
    feature_dim=int(feature_dim),
    hidden_size=128,
    num_layers=3,
    bidirectional=True,
    company_count=len(company_list),
    company_emb_dim=32,
    dropout=0.3
).to(DEVICE)

info("Model instance created. Loading weights...")

try:
    model.load_state_dict(checkpoint["model_state"])
    info("✅ Weights loaded successfully (strict load).")
except Exception as e:
    # Try a tolerant load (in case of minor key mismatches)
    try:
        model.load_state_dict(checkpoint["model_state"], strict=False)
        info(f"⚠️ Weights loaded with strict=False due to: {e}")
    except Exception as e2:
        info(f"❌ Failed to load weights: {e2}")
        raise SystemExit(1)

model.eval()
info("✅ Model set to eval mode")

# ===========================
# Step 6: Evaluate
# ===========================
info("\nStep 6: Running evaluation loop...")
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

true_vals = []
preds = []

with torch.no_grad():
    for i, (Xb, yb, cb) in enumerate(test_loader):
        info(f"  Evaluating batch {i+1}/{len(test_loader)} ...")
        Xb = Xb.to(DEVICE)
        cb = cb.to(DEVICE)
        try:
            out = model(Xb, cb)
            out_np = out.detach().cpu().numpy()
            preds.extend(out_np.tolist())
            true_vals.extend(yb.numpy().tolist())
        except Exception as e:
            info(f"    ❌ Error during model forward: {e}")
            # continue to next batch

info(f"\n✅ Collected {len(preds)} predictions and {len(true_vals)} ground-truth values")

if len(preds) == 0:
    info("❌ No predictions were produced. Exiting.")
    raise SystemExit(1)

# ===========================
# Step 7: Compute metrics
# ===========================
info("\nStep 7: Computing metrics...")
mse = mean_squared_error(true_vals, preds)
mae = mean_absolute_error(true_vals, preds)
rmse = np.sqrt(mse)
r2 = r2_score(true_vals, preds)

info("\n📊 Test Results:")
info(f"  MSE  = {mse:.6f}")
info(f"  RMSE = {rmse:.6f}")
info(f"  MAE  = {mae:.6f}")
info(f"  R²   = {r2:.6f}")
info("\n✅ Evaluation complete.")
